# **Rwanda HydroServer Exercise 1: Loading Sensor Data for a Monitoring Station**

Code adapted by the Center for Geospatial Solutions from examples developed by the HydroServer development team, including Jeff Horsburgh, Ken Lippold, and Daniel Slaugh. The original materials are available in this [HydroShare resource](https://www.hydroshare.org/resource/136c2bc6512540d59ce707bbd9c93e8a/).


### **Overview**

This exercise guides you through creating a HydroServer workspace, a monitoring site, and a stage datastream to manage telemetry water-level observations from the Kanzenze monitoring station (259501).

### **Description**

This Jupyter Notebook demonstrates how to load time series data for a monitoring site from a comma separate values (CSV) file into a datastream in HydroServer. It demonstrates the following:

1. Connects to HydroServer
2. Creates a Workspace
3. Creates a monitoring site (Thing)
4. Creates 3 Observed Properties (measured variables)
5. Creates 3 Units for the Observed Properties
6. Creates a new Sensor
7. Creates a new Processing Level
8. Creates 3 Datastreams for the Observed Properties
9. Loads data from a CSV file into the 3 Datastreams

More detailed examples of how to use hydroserverpy are available in HydroServer's documentation at:

* https://hydroserver2.github.io/hydroserver/user-guides/how-to/using-the-python-client.html
* https://www.hydroserver.org

### **Prerequisites**

You must have an account on the HydroServer Playground instance to run this notebook. If you haven't set up your user account yet, navigate to https://playground.hydroserver.org and follow the instructions to create a new user account.

### **Software Requirements**

This notebook was developed using Python Verion 3.14 and Version 1.11 of the hydroserverpy Python package.

## 1. **Getting Started**

---

### **Verify hydroserverpy is installed correctly**

The Python environment set up for this workshop already has hydroserverpy version 1.9 installed. If you downloaded this notebook to run locally or you are running in a different JupyterHub environment, you should uncomment the final line in the following code block and run it to ensure that the right version of the hydroserverpy package is installed. The hydroseverpy package version needs to match the version of HydroServer installed on the instance you are connecting to. For this example, we'll be using the Playground instance of HydroServer at [https://playground.hydroserver.org](https://playground.hydroserver.org), which is using Version 1.11.

In [1]:
# Make sure the correct version of hydroserverpy is installed
# The playground instance of hydroserver is using Version 1.9
# Uncomment and run the following line of code if you need to verify the installation of hydroserverpy
!pip install hydroserverpy==1.11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.8/114.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 12.3 MB/s eta 0:00:00


### **Import Required Packages**

For this example, we'll use hydroserverpy, pandas, and the datetime packages.

In [2]:
from hydroserverpy import HydroServer
import pandas as pd
from datetime import datetime
from getpass import getpass

### Set the Initial Parameters to Connect to HydroServer

The first step in interacting with a HydroServer instance is to create a connection to that instance. There's two ways to do this:
1. You can create a Workspace in the HydroServer Data Management App and then create an API key that allows you to connect to HydroServer using that API key.
2. You can use your HydroServer username and password to connect.

For this example, we will use your username and password because we will create the Workspace in code. To use an API key, you must create the Workspace first and then create an API key in the HydroServer Data Management App.

**IMPORTANT: In the following code, change the email to match the HydroServer user account you created.**

In [3]:
# Set initial parameters to connect to HydroServer
hydroserver_host = 'https://playground.hydroserver.org'

# Change the email and password below to your HydroServer username and password
hydroserver_email = 'svicario@lincolninst.edu' #'user@youremail.com'
hydroserver_password = getpass('Enter your HydroServer password: ') #getpass('Enter your HydroServer password: ')

Enter your HydroServer password: ··········


### **Initialize HydroServer Connection**

Initialize the connection to HydroServer with the connection information specified above.

In [4]:
# Initialize HydroServer connection with credentials.
hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password
)

print('\nSuccessfully connected to HydroServer!')


Successfully connected to HydroServer!


### **Create a Workspace**

Now that we are connected to HydroServer, we can create a Workspace for loading some data. The Workspace is an organizing container within which you will create monitoring Sites, Datastreams, and other metadata and content. Access control is also applied at the Workspace level in HydroServer. You can give read-only or edit permission to other HydroServer users to work in your Workspace.

**IMPORTANT**: In the following cell, change the name of the workspace using your name so it is unique to you.

In [5]:
# Change the workspace name here using your first name so it is unique to you
workspace_name = "Rwanda Training 2026"  # Choose a unique workspace name
new_workspace = hs.workspaces.create(name=workspace_name, is_private=False)

# Get the UUID of the workspace you just created - we'll use it later
workspace_id = new_workspace.uid

# Print some properties of the workspace
print(f"Created workspace named {new_workspace.name}")
print(f"Workspace ID: {workspace_id}")

Created workspace named Rwanda Training 2026
Workspace ID: 019ffae0-8a1d-7c6f-add4-f52cdcc3a2df


## 2. **Set Up Metadata for Loading Data**

---

Now that you have a Workspace, you can create objects and their metadata inside that workspace in preparation for loading data. In the following sections, we will create the necessary metadata to load data for a time series of observations for a monitoring site. You can create this metadata using the web user interface of the Data Management App, or you can do it using code, which we are demonstrating here. Code can make the process faster and more repeatable.

HydroServer uses a modified version of the OGC SensorThings API data model for storing time series data and their associated metadata. HydroServer's data model includes the following important entities that we need to create before loading data:

* **Thing**: A monitoring station on which or at which data were collected (e.g., a streamflow gage or weather station).
* **Observed Property**: The variable that is measured (e.g., discharge, water temperature, etc.).
* **Units of Measure**: The units of measure associated with the Observation values (e.g, cubic meters per second).
* **Sensor**: The instrument or method used to measure or create the Observation values.
* **Processing Level**: The degree of processing that has been applied to the Observation values (e.g., "Raw" or "Quality Controlled").
* **Datastream**: A description of the time series that includes all of these attributes.

Once all of these metadata tables have been populated, the time series of data values can be loaded to the **Observations** table in the database.

**NOTE**: To create objects in HydroServer, you will have to pass their required and optional metadata elements. For more information about HydroServer's data model and a data dictionary that describes each of the entities, see HydroServer's documentation at https://www.hydroserver.org.

### **Create a Monitoring Site (Thing)**

First, we'll create a monitoring site. The OGC SensorThings API used by HydroServer calls this a "Thing". We are going to load data for a monitoring site at Kanzenze Bridge located on the Akagera River, between Kicukiro and Bugesera districts in Rwanda. So, we need to specify the metadata for the monitoring site we are creating. Once the monitoring site is created, we can access its landing page in HydroServer by building a URL using its UUID.

In [6]:
# Create a Thing for the Kanzenze Hydrological Station

new_thing = hs.things.create(
    workspace=workspace_id,
    name='Kanzenze Hydrological Station',
    description='Hydrological monitoring station on the Nyabarongo River at Kanzenze, Rwanda.',
    sampling_feature_type='Site',
    sampling_feature_code='259501',
    site_type='Stream',
    elevation_m=1338.0,
    latitude=-2.0613,
    longitude=30.0877,
    admin_area_1='Eastern Province',
    admin_area_2='Bugesera',
    country='RW',
    data_disclaimer='Data provided by the Rwanda Water Resources Board (RWB).',
    is_private=False
)

# Get the ID for the new Thing and print its HydroServer landing page

thing_id = new_thing.uid

print(f'Created new thing with ID: {thing_id}')
print('You can access the new Thing in the HydroServer Data Management App at:')
print(f'{hydroserver_host}/sites/{thing_id}')

Created new thing with ID: 019ffae0-9671-78f9-8fe1-1ef91d34b905
You can access the new Thing in the HydroServer Data Management App at:
https://playground.hydroserver.org/sites/019ffae0-9671-78f9-8fe1-1ef91d34b905


### **Create a Sensor**

The OGC SensorThings API data model refers to the method used for creating observations as the "Sensor". In many cases this will be a physical sensor installed at the monitoring site. But, sometimes other methods are used to create observations. We need to create the metadata describing this so potential data users know how the data were created.

**NOTE**: The specific metadata required when creating metadata for a Sensor is dependent upon the "Method Type". For instrument deployments, specific information about the manufacturer and model of the sensor should be specified. For "Derivation" methods, the name and description are required, and a method_code and method_link can be specified if needed.

In [7]:
historical_stage_sensor = hs.sensors.create(
    workspace=workspace_id,
    name='Kanzenze Historical Stage Observations',
    description='Historical stage observations recorded at the Kanzenze hydrological station.',
    encoding_type='application/json',
    method_type='Observation',
    method_code='kanzenze-historical-stage'
)

### **Create an Observed Property**

Observed Properties are the variables measured at the monitoring site. Similar to creating the monitoring site (Thing), we need to know the required and optional metadata elements for Observed Properties so we can pass them to the ```create()``` method.For this example, we will load one time series from a CSV file.

In [8]:
stage = hs.observedproperties.create(
    workspace=workspace_id,
    name='Stage',
    definition='Stage',
    description='Stage is the height of the water surface at a monitoring location relative to a reference level.',
    observed_property_type='Hydrology',
    code='Stage'
)

print("Created observed property:")
print(f"{stage.name}: {stage.uid}")

Created observed property:
Stage: 019ffae0-a74d-7f2d-b76f-b35a9f8223c2


### **Create Units of Measure**

Next we need to add metadata to specify the Units of measure used for recording the data in the CSV file. We need one unit for each of the columns of data we are loading because they are all different.

In [9]:
stage_unit = hs.units.create(
    workspace=workspace_id,
    name='Meter',
    symbol='m',
    definition='Unit for water stage',
    unit_type='Length'
)

print("Created unit:")
print(f"{stage_unit.name}: {stage_unit.uid}")

Created unit:
Meter: 019ffae0-ae45-704a-86da-1b5522953b68


### **Create a Processing Level**

In HydroServer, the Processing Level indicates the degree of processing a datastream has been subject to. For example, data can be "Raw", which means that they were recorded in the field and nobody has looked at them yet, or they could be "Quality Controlled", which means that a technician has reviewed the data. All of the data we are loading right now are raw observations from the field with no processing, so we need a Processing Level that indicates this.

In [10]:
new_processing_level = hs.processinglevels.create(
    workspace=workspace_id,
    code='Raw',
    definition='Raw Data',
    explanation='Data that have not been processed or quality controlled.'
)

print("Created processing levels:")
print(f"{new_processing_level.code}: {new_processing_level.uid}")

Created processing levels:
Raw: 019ffae0-b42e-76bc-a635-5ca219fa6517


### **Create a Datastream**

The last step before loading data is to create metadata for the Datastream. This helps us link the time series values to where they were measured, which Observed Property they represent, which Units they are recorded in, etc. In the following code, we create the necessary datastream metadata, using the UUIDs for the other metadata entities we created above for the three datastreams we want to load data to.

**NOTE**: Since these Datastreams are new, they don't contain any Observation values yet. We'll set the ```value_count=0``` and arbitrarily set the ```phenomenon_begin_time``` and ```phenomenon_end_time```. Those will get reset when we load Observation values. All of the LRO aquatic sensor data have spacing of 15-minutes, so we set that accordingly. Each Datastream has a name and description that we'll set using some attributes of the Datastream, but you can name these accoding to your own naming conventions.

In [11]:
ds_stage = hs.datastreams.create(
    name=f"{stage.name} - Historical - {new_thing.name}",
    description=f'Historical {stage.name.lower()} observations at {new_thing.name}',
    thing=new_thing.uid,
    sensor=historical_stage_sensor.uid,
    observed_property=stage.uid,
    processing_level=new_processing_level.uid,
    unit=stage_unit.uid,
    observation_type='Field Observation',
    result_type='Timeseries',
    sampled_medium='Surface Water',
    no_data_value=-9999,
    aggregation_statistic='Continuous',
    time_aggregation_interval=0,
    time_aggregation_interval_unit='minutes',
    intended_time_spacing=1,
    intended_time_spacing_unit='days',
    status='Complete',
    value_count=0,
    phenomenon_begin_time=datetime(year=1971, month=3, day=7),
    phenomenon_end_time=datetime(year=2015, month=6, day=8),
    is_private=False,
    is_visible=True
)

print("Created datastream:")
print(f"{ds_stage.name}: {ds_stage.uid}")

Created datastream:
Stage - Historical - Kanzenze Hydrological Station: 019ffae0-ba46-7eca-a18e-0cab5d9c3248


## 3. Load Time Series Data from the CSV File

---

Now that we have all of the metadata we need, we can read the CSV data file and load data. The following sections break this down using a convenience data structure for mapping the names of the columns in the CSV file to the datastreams they represent and then chunking up the CSV data to load it into HydroServer. The UUIDs for the datastreams come from the datastreams we just created in the last step.

### **Specify the CSV File to Load Data From**

For this example, we are going to load data to a monitoring site we create in HydroServer from a comma-separated values (CSV) file. This functionality could be used to load data from any CSV file that has a single datetime column and then any number of data columns (e.g., a log file from a field datalogger). To limit the size of the requests we are making to the HydroServer API, we will also specify a chunk size for loading data. We'll load the whole file, but in requests that are sized by the chunk size we set here.

In [27]:
file_to_load = (
    "https://raw.githubusercontent.com/"
    "savicario/addis-ababa-hydroserver-training-2026/"
    "main/rwanda/Exercise1/data/RWANDA_RWB_2026-08-11.csv"
)

chunk_size = 10000

print(file_to_load)

https://raw.githubusercontent.com/savicario/addis-ababa-hydroserver-training-2026/main/rwanda/Exercise1/data/RWANDA_RWB_2026-08-11.csv


### **Create Datastream UUID Mapping to CSV File Columns**

 We'll first create a convenience data structure (a Python List object) to match up the column names in the CSV file with the UUIDs of the Datastreams we need to load their data into. We'll use this to specify which columns from the file we want to load. Each element in the Python list will be a Tuple specifying the name of the column in the CSV file and the UUID for the datastream the values in that column should be loaded into.

**NOTE**: The datastream metadata must exist in HydroServer before loading data into it. You must have one entry in this list for each CSV column/datastream to be loaded into the database.

In [28]:
#
datastreams = [
    ('Value', ds_stage.uid)
]

print(f"Number of datastreams to load: {len(datastreams)}")

Number of datastreams to load: 1


### **Read the CSV File Using Pandas**

Read the CSV file containing data to load using Pandas. Skip comment rows in the header that don't contain data or column headings and also parse the dates in the LocalDateTime and DateTimeUTC columns to ensure that we datetime objects instead of strings in the resulting Pandas dataframe.

In [29]:
file_to_load = (
    "https://raw.githubusercontent.com/"
    "savicario/addis-ababa-hydroserver-training-2026/"
    "main/rwanda/Exercise1/data/RWANDA_RWB_2026-08-11.csv"
)

# Read the CSV data file containing data to load.

df = pd.read_csv(
    file_to_load,
    sep=',',

    # Skip the 8 metadata rows at the beginning of the Rwanda CSV file.
    skiprows=8,

    # Use "Timestamp,Value" as the column names after skipping the metadata rows.
    header=0,
    parse_dates=['Timestamp'],

    low_memory=False
)

print('Data file read successfully.')
print(f"Number of rows in CSV file to be loaded: {len(df)}")

Data file read successfully.
Number of rows in CSV file to be loaded: 10641


### **Specify the Time Zone for the Data**

LRO datalogger timestamps are recorded in Mountain Standard Time (UTCOffset = -7 hours). If no timezone offset information is provided when loading data, HydroServer will assume the timestamps you provide are in UTC. The data file we are loading data from already has a DateTimeUTC column, but the fact that it is UTC is not encoded in the file. So, we need to tell Pandas that those timestamps are UTC before loading the data.

In [30]:
# Convert the Kanzenze timestamps from their recorded timezone (+02:00) to UTC.
df['Timestamp'] = pd.to_datetime(df['Timestamp'], utc=True)

print('Timestamps in the "Timestamp" column have been converted to UTC.')

Timestamps in the "Timestamp" column have been converted to UTC.


### **Chunk the Data and Load into HydroServer**

The last step in loading data is to divide it up into reasonably sized chunks to load it into HydroServer. We set the chunk size (the number of records to add at one time) at the top of this notebook. The code below just divides the Pandas dataframe up into chunks for each datastream according to the chunk size and then loads the data one chunk at a time using the hydroserverpy ```load_observations()``` function.

In [31]:
# Select the Timestamp and Stage value columns
observations = df[['Timestamp', 'Value']]

# Rename columns to the names expected by hydroserverpy
observations = observations.rename(
    columns={
        'Timestamp': 'phenomenon_time',
        'Value': 'result'
    }
)

# Get the Kanzenze Historical Stage datastream
datastream = hs.datastreams.get(uid=ds_stage.uid)

# Upload the observations to HydroServer
datastream.load_observations(observations)

print(f"Loaded {len(observations)} Stage observations.")
print(f"Access the data in HydroServer at: {hydroserver_host}/sites/{thing_id}")

Loaded 10641 Stage observations.
Access the data in HydroServer at: https://playground.hydroserver.org/sites/019ffae0-9671-78f9-8fe1-1ef91d34b905


## **Summary**

Now you know the general patterns for how to create a workspace in HydroServer and then how to create metadata and then load data into HydroServer from a CSV file using hydroserverpy.